# Lab 1 — Docker + FastAPI

**Goal:** Wrap your agent in a production FastAPI server and containerise it with Docker.

**Why:** HuggingFace Spaces is great for demos. But enterprise deployments need:
- A standard REST API that any frontend or service can call
- A Docker container that runs identically in dev, staging, and production
- Health checks so load balancers know if the service is up
- Structured JSON responses (not just a chatbox)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
print('Week 12 ready ✓')

## 1. Inspect the FastAPI server
Read `agent_server.py` to understand the API design.

In [ ]:
with open('agent_server.py') as f:
    print(f.read())

## 2. Start the server (in a background thread for notebook use)

In [ ]:
import threading, time, subprocess, sys

# Start the FastAPI server in a subprocess
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'agent_server:app', '--port', '8001', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)  # wait for startup
print('Server started (PID:', server_process.pid, ')')

## 3. Test the API

In [ ]:
import requests

BASE_URL = 'http://localhost:8001'

# Health check
health = requests.get(f'{BASE_URL}/health')
print('Health:', health.json())

In [ ]:
# Chat
response = requests.post(f'{BASE_URL}/chat', json={
    'message': 'What are the benefits of containerisation?',
    'user_id': 'student_001'
})
print('Status:', response.status_code)
print('Response:', response.json())

In [ ]:
# Admin budget endpoint
budget = requests.get(f'{BASE_URL}/admin/budget', headers={'x-api-key': 'admin'})
print('Budget:', budget.json())

In [ ]:
# Audit log
audit = requests.get(f'{BASE_URL}/admin/audit', headers={'x-api-key': 'admin'})
print('Audit:', audit.json())

## 4. Review the Dockerfile

In [ ]:
with open('Dockerfile') as f:
    print(f.read())

print('\nBuild command:')
print('  docker build -t enterprise-agent:latest .')
print('\nRun command:')
print('  docker run -p 8000:8000 -e OPENAI_API_KEY=sk-... enterprise-agent:latest')

In [ ]:
# Cleanup
server_process.terminate()
print('Server stopped.')

## Exercise
Add a `/chat/stream` endpoint to the FastAPI server that streams the LLM response using Server-Sent Events (SSE).  
**Hint:** Use `openai.stream=True` and FastAPI's `StreamingResponse` with `media_type='text/event-stream'`.

In [ ]:
# Your code here
